In [2]:
# Intialization
import os
import sys

os.environ["SPARK_HOME"] = "/home/talentum/spark"
os.environ["PYLIB"] = os.environ["SPARK_HOME"] + "/python/lib"
# In below two lines, use /usr/bin/python2.7 if you want to use Python 2
os.environ["PYSPARK_PYTHON"] = "/usr/bin/python3.6" 
os.environ["PYSPARK_DRIVER_PYTHON"] = "/usr/bin/python3"
sys.path.insert(0, os.environ["PYLIB"] +"/py4j-0.10.7-src.zip")
sys.path.insert(0, os.environ["PYLIB"] +"/pyspark.zip")

# NOTE: Whichever package you want mention here.
# os.environ['PYSPARK_SUBMIT_ARGS'] = '--packages com.databricks:spark-xml_2.11:0.6.0 pyspark-shell' 
# os.environ['PYSPARK_SUBMIT_ARGS'] = '--packages org.apache.spark:spark-avro_2.11:2.4.0 pyspark-shell'
os.environ['PYSPARK_SUBMIT_ARGS'] = '--packages com.databricks:spark-xml_2.11:0.6.0,org.apache.spark:spark-avro_2.11:2.4.3 pyspark-shell'
# os.environ['PYSPARK_SUBMIT_ARGS'] = '--packages com.databricks:spark-xml_2.11:0.6.0,org.apache.spark:spark-avro_2.11:2.4.0 pyspark-shell'

In [3]:
#Entrypoint 2.x
from pyspark.sql import SparkSession
import pyspark.sql.functions as F

spark = SparkSession.builder.appName("Spark SQL basic example").enableHiveSupport().getOrCreate()

# On yarn:
# spark = SparkSession.builder.appName("Spark SQL basic example").enableHiveSupport().master("yarn").getOrCreate()
# specify .master("yarn")

sc = spark.sparkContext

In [4]:
df = spark.read.format("csv").option("header", "True").load("file:///home/talentum/Big_Data_Project_Work/Big_Data_Project/2015.csv")

In [6]:
print(df.count())
print(len(df.columns))

5819079
28


In [7]:
df.printSchema()

root
 |-- FL_DATE: string (nullable = true)
 |-- OP_CARRIER: string (nullable = true)
 |-- OP_CARRIER_FL_NUM: string (nullable = true)
 |-- ORIGIN: string (nullable = true)
 |-- DEST: string (nullable = true)
 |-- CRS_DEP_TIME: string (nullable = true)
 |-- DEP_TIME: string (nullable = true)
 |-- DEP_DELAY: string (nullable = true)
 |-- TAXI_OUT: string (nullable = true)
 |-- WHEELS_OFF: string (nullable = true)
 |-- WHEELS_ON: string (nullable = true)
 |-- TAXI_IN: string (nullable = true)
 |-- CRS_ARR_TIME: string (nullable = true)
 |-- ARR_TIME: string (nullable = true)
 |-- ARR_DELAY: string (nullable = true)
 |-- CANCELLED: string (nullable = true)
 |-- CANCELLATION_CODE: string (nullable = true)
 |-- DIVERTED: string (nullable = true)
 |-- CRS_ELAPSED_TIME: string (nullable = true)
 |-- ACTUAL_ELAPSED_TIME: string (nullable = true)
 |-- AIR_TIME: string (nullable = true)
 |-- DISTANCE: string (nullable = true)
 |-- CARRIER_DELAY: string (nullable = true)
 |-- WEATHER_DELAY: strin

In [8]:
print(df.columns)

['FL_DATE', 'OP_CARRIER', 'OP_CARRIER_FL_NUM', 'ORIGIN', 'DEST', 'CRS_DEP_TIME', 'DEP_TIME', 'DEP_DELAY', 'TAXI_OUT', 'WHEELS_OFF', 'WHEELS_ON', 'TAXI_IN', 'CRS_ARR_TIME', 'ARR_TIME', 'ARR_DELAY', 'CANCELLED', 'CANCELLATION_CODE', 'DIVERTED', 'CRS_ELAPSED_TIME', 'ACTUAL_ELAPSED_TIME', 'AIR_TIME', 'DISTANCE', 'CARRIER_DELAY', 'WEATHER_DELAY', 'NAS_DELAY', 'SECURITY_DELAY', 'LATE_AIRCRAFT_DELAY', 'Unnamed: 27']


<h3> Data Cleaning: Remove Unnecessary Column </h3>

<h3>
The dataset contains an extra column named `Unnamed: 27`, which does not hold any meaningful information and was created due to formatting issues during CSV generation.

This column is removed to improve data quality and reduce unnecessary storage and processing overhead.</h3>

In [14]:
df = df.drop("Unnamed: 27")

<h3>
     Data Type Standardization

When the CSV file is loaded into Spark, several numerical fields are interpreted as string values.

To enable mathematical operations, aggregations, statistical analysis, and delay calculations, the following columns are converted to numeric (double) datatype:

- DEP_DELAY
- ARR_DELAY
- DISTANCE
- CARRIER_DELAY
- WEATHER_DELAY
- NAS_DELAY
- SECURITY_DELAY
- LATE_AIRCRAFT_DELAY

This step ensures accurate analytical processing in subsequent stages.
</h3>

In [15]:
from pyspark.sql.functions import col

numeric_cols = [
    "DEP_DELAY",
    "ARR_DELAY",
    "DISTANCE",
    "CARRIER_DELAY",
    "WEATHER_DELAY",
    "NAS_DELAY",
    "SECURITY_DELAY",
    "LATE_AIRCRAFT_DELAY"
]

for c in numeric_cols:
    df = df.withColumn(c, col(c).cast("double"))

<h3>
 Data Quality Filtering

To maintain data integrity, unrealistic departure delay values are removed.

Business Rules:
- Minimum valid delay = -30 minutes
- Maximum valid delay = 1800 minutes

Records outside this range are considered anomalies and excluded from further analysis.
</h3>

In [16]:
df = df.filter(
    (col("DEP_DELAY") >= -30) &
    (col("DEP_DELAY") <= 1800)
)

<h3>
 Data Quality Assessment: Null Value Analysis

A null value analysis is performed to understand data completeness across all attributes.

The objective is to:

- Identify missing values
- Assess dataset quality
- Determine whether missing values are expected or indicate data issues
- Support future cleaning and preprocessing decisions

This analysis forms part of the Data Quality Gate stage of the pipeline.
</h3>

In [17]:
from pyspark.sql.functions import count, when

df.select([
    count(
        when(col(c).isNull(), c)
    ).alias(c)
    for c in df.columns
]).show()

+-------+----------+-----------------+------+----+------------+--------+---------+--------+----------+---------+-------+------------+--------+---------+---------+-----------------+--------+----------------+-------------------+--------+--------+-------------+-------------+---------+--------------+-------------------+
|FL_DATE|OP_CARRIER|OP_CARRIER_FL_NUM|ORIGIN|DEST|CRS_DEP_TIME|DEP_TIME|DEP_DELAY|TAXI_OUT|WHEELS_OFF|WHEELS_ON|TAXI_IN|CRS_ARR_TIME|ARR_TIME|ARR_DELAY|CANCELLED|CANCELLATION_CODE|DIVERTED|CRS_ELAPSED_TIME|ACTUAL_ELAPSED_TIME|AIR_TIME|DISTANCE|CARRIER_DELAY|WEATHER_DELAY|NAS_DELAY|SECURITY_DELAY|LATE_AIRCRAFT_DELAY|
+-------+----------+-----------------+------+----+------------+--------+---------+--------+----------+---------+-------+------------+--------+---------+---------+-----------------+--------+----------------+-------------------+--------+--------+-------------+-------------+---------+--------------+-------------------+
|      0|         0|                0|     0| 

<h3>
 Delay Severity Classification

Flights are categorized based on departure delay duration.

Categories:

| Delay Range | Category |
|-------------|-----------|
| < 15 min | ON_TIME |
| 15 - 59 min | MINOR_DELAY |
| 60 - 179 min | MAJOR_DELAY |
| >= 180 min | CRITICAL_DELAY |

This classification simplifies downstream reporting and dashboard visualizations.
</h3>

In [18]:
from pyspark.sql.functions import when

df = df.withColumn(
    "delay_category",
    when(col("DEP_DELAY") < 15, "ON_TIME")
    .when(col("DEP_DELAY") < 60, "MINOR_DELAY")
    .when(col("DEP_DELAY") < 180, "MAJOR_DELAY")
    .otherwise("CRITICAL_DELAY")
)

<h3>
Exploratory Data Analysis

A sample of flight records is displayed to verify:

- Origin Airport
- Destination Airport
- Departure Delay
- Arrival Delay
- Late Aircraft Delay

This helps validate data quality and understand the structure of operational flight information.
</h3>

In [9]:
df.select(
    "ORIGIN",
    "DEST",
    "DEP_DELAY",
    "ARR_DELAY",
    "LATE_AIRCRAFT_DELAY"
).show(20, False)

+------+----+---------+---------+-------------------+
|ORIGIN|DEST|DEP_DELAY|ARR_DELAY|LATE_AIRCRAFT_DELAY|
+------+----+---------+---------+-------------------+
|MCO   |FLL |-4.0     |-5.0     |null               |
|LGA   |FLL |14.0     |-1.0     |null               |
|FLL   |MCO |12.0     |16.0     |0.0                |
|IAH   |LAS |11.0     |-9.0     |null               |
|IAH   |ORD |-3.0     |-15.0    |null               |
|FLL   |STT |5.0      |14.0     |null               |
|DFW   |BWI |-5.0     |-19.0    |null               |
|BOS   |PBI |-1.0     |-5.0     |null               |
|PBI   |BOS |-1.0     |-1.0     |null               |
|ORD   |OAK |0.0      |-37.0    |null               |
|STT   |FLL |1.0      |-6.0     |null               |
|LAS   |DFW |-1.0     |-1.0     |null               |
|PHL   |DFW |-8.0     |4.0      |null               |
|LAX   |IAH |-7.0     |14.0     |null               |
|IAH   |LAX |-1.0     |-12.0    |null               |
|FLL   |ACY |15.0     |18.0 

<h3>
Delay Propagation Impact Dataset

The dataset contains a field called `LATE_AIRCRAFT_DELAY`.

This field indicates delays caused by late-arriving aircraft from previous operations.

Flights with non-null values in this field are extracted into a dedicated propagation analysis dataset for further investigation.
</h3>

In [10]:
from pyspark.sql.functions import col

propagation_df = df.filter(
    col("LATE_AIRCRAFT_DELAY").isNotNull()
)

print(propagation_df.count())

1063439


<h3>
Airport Delay Propagation Analysis

Average Late Aircraft Delay is calculated for each origin airport.

Objective:

- Identify airports most affected by propagated delays
- Measure downstream operational impact
- Detect potential bottlenecks within the aviation network

Higher values indicate stronger delay propagation effects.
</h3>

In [11]:
from pyspark.sql.functions import avg

propagation_df.groupBy("ORIGIN") \
    .agg(
        avg("LATE_AIRCRAFT_DELAY")
        .alias("avg_propagation_delay")
    ) \
    .orderBy(
        "avg_propagation_delay",
        ascending=False
    ) \
    .show(20, False)

+------+---------------------+
|ORIGIN|avg_propagation_delay|
+------+---------------------+
|VEL   |77.14285714285714    |
|APN   |62.31645569620253    |
|OTH   |59.96511627906977    |
|PLN   |55.38834951456311    |
|BQN   |51.68402777777778    |
|MEI   |49.95631067961165    |
|ACT   |49.84981684981685    |
|BTM   |48.51162790697674    |
|WRG   |47.669291338582674   |
|ESC   |47.472727272727276   |
|STC   |45.75                |
|HIB   |45.131147540983605   |
|MKG   |44.207142857142856   |
|PAH   |43.06741573033708    |
|YAK   |42.92307692307692    |
|ACV   |42.751612903225805   |
|BRW   |42.696296296296296   |
|PSG   |42.651315789473685   |
|MBS   |42.248366013071895   |
|EGE   |42.05673758865248    |
+------+---------------------+
only showing top 20 rows



<h3>
Delay Propagation Rate Calculation

The percentage of flights affected by propagated delays is calculated.

Formula:

Propagation Rate =
(Number of Flights with Late Aircraft Delay > 0)
/
(Total Number of Flights)
× 100

This metric quantifies the overall impact of delay propagation across the aviation network.
</h3>

In [12]:
from pyspark.sql.functions import count, when, col

total = df.count()

propagated = df.filter(
    col("LATE_AIRCRAFT_DELAY") > 0
).count()

print(propagated / total * 100)

9.571153785676394


<h3>
# Delay Propagation Summary Statistics

Key metrics generated:

- Total Flights Processed
- Flights Affected by Delay Propagation
- Overall Propagation Rate (%)

These metrics provide a high-level operational view of propagation-related disruptions and serve as key performance indicators (KPIs) for dashboard reporting.
</h3>

In [13]:
total = df.count()

propagated = df.filter(
    col("LATE_AIRCRAFT_DELAY") > 0
).count()

print("Total Flights:", total)
print("Propagation Flights:", propagated)
print("Propagation Rate:", round((propagated/total)*100,2))

Total Flights: 5819079
Propagation Flights: 556953
Propagation Rate: 9.57
